# A Bio-Image Analysis chatbot GUI

In [1]:
# This Jupyter notebook can be used to build a graphical user interface for a chatbot. 
# Just run this from the terminal:
#
# voila gui.ipynb

In [2]:
import openai

In [3]:
def prompt_ollama(prompt:str, model="gemma3:1b"):
    """A prompt helper function that sends a prompt to 
    ollama and returns only the text response."""
    # handle longer conversions
    if isinstance(prompt, str):
        messages=[{"role": "user", "content": prompt}]
    else:
        messages = prompt
    
    # setup connection to the LLM server
    client = openai.OpenAI(
        base_url = "http://localhost:11434/v1",
        api_key = "none" # not required by ollama
    )
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    
    # extract answer
    return response.choices[0].message.content

In [4]:
prompt = prompt_ollama

In [5]:
chat_history = []
def prompt_with_memory(message:str):
    
    # convert message in the right format and store it in memory
    question = {"role": "user", "content": message}
    chat_history.append(question)
    
    # receive answer
    response = prompt(chat_history)
    
    # convert answer in the right format and store it in memory
    answer = {"role": "assistant", "content": message}
    chat_history.append(answer)
    
    return response

In [6]:
# Here we can inject knowledge into the chatbot, by telling things the user of our chatbot won't see.

_ = prompt_with_memory("""
You are an extremely talented Bio-image analyst, but you cannot code.
You are an excellent theoretician who knows which algorithm or software to use.
You use preferably classical image processing algorithms.

You use the word "ehm" at random places in sentences. Every second sentence contains "ehm" or "ehhh".
You are also rude and pedantic. You tend to tell everyone that you know things better than everybody else.
Keep your answers 2-3 sentences short.
""")

In [7]:
import ipywidgets as widgets
from IPython.display import display

# Create widgets
output_label = widgets.HTML(value="")
text_input = widgets.Text(placeholder="Type something here...")
submit_button = widgets.Button(description="Submit")

def on_submit(e):
    question = text_input.value
    if len(question) == 0:
        return
    text_input.value = ""

    # submit prompt to LLM
    answer = prompt_with_memory(question)

    # Append question and answer to the existing HTML content
    output_label.value += f"""
    <div style='text-align:right; color: blue; font-size: 20px'>{question}</div>
    <div style='text-align:left; color: darkgreen; font-size: 20px'>{answer}</div>
    """

# Attach the event handler to the text field and the button
text_input.continuous_update = False
text_input.observe(on_submit)
submit_button.on_click(on_submit)

# Arrange the widgets for display
display(output_label, widgets.HBox([text_input, submit_button]))

HTML(value='')